# Lab 3.3 — Graph Attention Networks
**Module III · Graph Neural Networks**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanielFPerez/llm-gnns-course_solutions/blob/main/module-3-gnn/lab3_3_attention_gat.ipynb)

---

## What you will do
1. Understand **graph attention** — how GAT differs from GCN by learning per-edge importance weights.
2. Build and train a **2-layer GAT** on Cora and compare accuracy with the GCN baseline.
3. **Extract attention weights** from a trained GAT and interpret them: which neighbours does each node focus on?
4. Visualise a **1-hop attention subgraph** around a specific node — showing edge thickness proportional to attention.
5. `[Extension]` Connect GAT attention to the Transformer self-attention you studied in Module II.

## Prerequisites
Labs 3.1 and 3.2 completed.

**Estimated time:** 55–70 min

---
## 0 · Setup

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab. Cloning the course solutions repository and installing dependencies...")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/DanielFPerez/llm-gnns-course_solutions.git"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r",
         "llm-gnns-course_solutions/environment/requirements.txt"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "torch-geometric"], check=True)
    sys.path.insert(0, "llm-gnns-course_solutions")
else:
    sys.path.insert(0, str(Path("..").resolve()))

print("Setup complete.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.nn import GATConv

from utils import plot_training_curves, plot_embeddings, plot_attention_subgraph, check_gnn_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

CLASS_NAMES = [
    "Case-Based", "Genetic Algorithms", "Neural Networks",
    "Probabilistic Methods", "Reinforcement Learning",
    "Rule Learning", "Theory",
]

In [ ]:
dataset = Planetoid(root="/tmp/Cora", name="Cora",
                    transform=T.NormalizeFeatures())
data = dataset[0].to(DEVICE)
print(data)

---
## 1 · From GCN to GAT: adding attention

**GCN recap:** In GCN, the aggregation weight for an edge $(j \to i)$ is fixed by the degree:
$$w_{ji} = \frac{1}{\sqrt{\deg(i) \cdot \deg(j)}}$$

Every neighbour contributes in proportion to its degree — there is no room for "this neighbour is more relevant than that one."

**GAT (Veličković et al., ICLR 2018)** replaces fixed weights with **learnable attention coefficients** $\alpha_{ji}$:

$$e_{ji} = \text{LeakyReLU}\!\left( \mathbf{a}^\top [W h_j \| W h_i] \right)$$
$$\alpha_{ji} = \text{softmax}_j(e_{ji}) = \frac{\exp(e_{ji})}{\sum_{k \in \mathcal{N}(i)} \exp(e_{ki})}$$
$$h_i' = \sigma\!\left( \sum_{j \in \mathcal{N}(i)} \alpha_{ji} W h_j \right)$$

Where $\|$ denotes concatenation and $\mathbf{a}$ is a learnable vector. In practice, **multi-head attention** (like in Transformers) runs $K$ independent attention mechanisms and concatenates (or averages) their outputs.

**Key insight:** The attention coefficient $\alpha_{ji}$ is a *data-dependent* weight — the model learns to focus more on informative neighbours and less on irrelevant ones. This is exactly the same idea as the Transformer's self-attention (Module II, Lab 2.1), applied to graph neighbourhoods.

### Exercise 3.3.1 `[Core]` — Build the GAT

Implement `GAT` as a `nn.Module` with:
- **Layer 1:** `GATConv(in_channels, hidden_channels, heads=8, dropout=0.6)` — 8 attention heads, each with `hidden_channels` output dimensions (total output: `8 × hidden_channels`).
- ELU activation + dropout on the input features.
- **Layer 2:** `GATConv(hidden_channels * 8, out_channels, heads=1, concat=False, dropout=0.6)` — single head, no concatenation (averages across heads → outputs `out_channels`).

The `forward(x, edge_index)` method passes `edge_index` to both layers.

**Note:** The original GAT paper applies dropout to the **input features** (not just between layers). Follow that convention.

In [ ]:
# YOUR CODE HERE
# Hint: Implement GAT as a nn.Module with:

class GAT(nn.Module):
    pass  # replace this

> **Note on hidden size:** The hidden dimension per head is `8`, so the layer-1 output is `8 × 8 = 64` dimensions — the same as the GCN. This makes the comparison fair in terms of model capacity.

### Exercise 3.3.2 `[Core]` — Train the GAT

Train the GAT for 200 epochs with `lr=0.005` and `weight_decay=5e-4`. Then:
1. Plot the training curves.
2. Print test accuracy and compare with the GCN from Lab 3.2.
3. Run `check_gnn_model("3.3.2", gat, data, split="test", min_acc=0.80)`.

In [ ]:
def train_epoch(model, data, optimiser):
    model.train()
    optimiser.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimiser.step()
    return loss.item()


@torch.no_grad()
def evaluate(model, data):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    results = {}
    for split in ["train", "val", "test"]:
        mask = getattr(data, f"{split}_mask")
        correct = (pred[mask] == data.y[mask]).sum().item()
        results[split] = correct / mask.sum().item()
    return results


def train_model(model, data, n_epochs=200, lr=0.01, weight_decay=5e-4):
    optimiser = torch.optim.Adam(model.parameters(), lr=lr,
                                 weight_decay=weight_decay)
    history = {"train_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(model, data, optimiser)
        metrics = evaluate(model, data)
        history["train_loss"].append(loss)
        history["train_acc"].append(metrics["train"])
        history["val_acc"].append(metrics["val"])

        if epoch % 50 == 0:
            print(f"Epoch {epoch:>3} | loss {loss:.4f} | "
                  f"train acc {metrics['train']:.3f} | val acc {metrics['val']:.3f}")

    test_metrics = evaluate(model, data)
    print(f"\n→ Final TEST accuracy: {test_metrics['test']:.3f}")
    return history, test_metrics

In [ ]:
# YOUR CODE HERE
# Hint: Train the GAT for 200 epochs with lr=0.005 and weight_decay=5e-4. Then:
raise NotImplementedError("Complete this exercise")

> **Result:** GAT achieves **~83–84% test accuracy** on Cora, versus ~81% for GCN — a small but consistent improvement. More importantly, the attention mechanism provides something GCN cannot: **interpretability**. We can inspect which neighbours the model attends to, giving us insight into *why* a classification was made. That is what we explore next.

---
## 2 · Extracting and interpreting attention weights

PyG's `GATConv` can return attention weights via `return_attention_weights=True`. This changes the return signature:

```python
(out, (edge_index_with_self_loops, alpha)) = conv(x, edge_index,
                                                   return_attention_weights=True)
```

- `edge_index_with_self_loops`: shape `(2, E')` — edges including self-loops added internally
- `alpha`: shape `(E', H)` — attention weight per edge per head

The attention weights are softmaxed per node — so for node $i$, the weights across its neighbours sum to 1.

### Exercise 3.3.3 `[Core]` — Extract attention weights from the first layer

Write a function `get_attention_weights(model, data)` that:
1. Calls the first GAT layer with `return_attention_weights=True`.
2. Returns `(edge_index, alpha)` where `alpha` has shape `(E', H)`.

Print the shape of both tensors and the minimum/maximum attention weight.

In [ ]:
# YOUR CODE HERE
# Hint: Write a function get_attention_weights(model, data) that:
raise NotImplementedError("Complete this exercise")

### Exercise 3.3.4 `[Core]` — Attention weight distribution

The distribution of attention weights tells us how focused the model is:
- If all $\alpha_{ji}$ for node $i$ are equal (uniform), the GAT is behaving like a plain mean aggregator — no attention.
- If weights are concentrated on 1–2 neighbours, the model is very selective.

Plot a histogram of all attention weights (head 0) using `plt.hist`. Is the distribution spread out or concentrated near 0? What does this imply?

In [ ]:
# YOUR CODE HERE
# Hint: The distribution of attention weights tells us how focused the model is:
raise NotImplementedError("Complete this exercise")

> **Observation:** The distribution is heavily skewed toward small values — most edges receive very low attention, while a small fraction of edges receive high attention. This confirms that the GAT has learned to be **selective**: it focuses on a few key neighbours rather than averaging all neighbours equally. The mean weight is ~1/mean_degree but there is substantial variance, indicating genuine learned focus.

---
## 3 · Visualising attention subgraphs

A powerful interpretability technique is to draw the **1-hop neighbourhood** of a target node with edges scaled by attention weight. Thick edges = high attention (the model focuses on this neighbour). This is one of the clearest ways to see what a GAT has learned.

### Exercise 3.3.5 `[Core]` — Visualise the attention subgraph of a node

Use `plot_attention_subgraph` from the `utils` module to draw the 1-hop neighbourhood of node **0** with edge widths proportional to head-0 attention weights.

```python
plot_attention_subgraph(
    edge_index=edge_index_attn,
    alpha=alpha,
    center_node=0,
    node_labels=data.y,
    label_names=CLASS_NAMES,
    head=0,
    title="GAT attention",
)
```

After viewing the plot:
1. Which neighbours receive the most attention?
2. Are the high-attention neighbours the same class as node 0?
3. What does this suggest about what the GAT has learned?

In [ ]:
# YOUR CODE HERE
# Hint: Use plot_attention_subgraph from the utils module to draw the 1-hop neighbourhood of node 0 with edge widths proporti...
raise NotImplementedError("Complete this exercise")

In [ ]:
# --- SOLUTION (analysis) ---
# Find the edges involving node 0
from_node0 = (src == 0).nonzero(as_tuple=True)[0]
to_node0   = (dst == 0).nonzero(as_tuple=True)[0]

print("\nAttention weights for edges FROM node 0 (head 0):")
for idx in from_node0:
    nbr = dst[idx].item()
    nbr_class = data.y[nbr].item()
    attn = alpha[idx, 0].item()
    same = "✓ same class" if nbr_class == node_class else "✗ diff class"
    print(f"  → node {nbr:>4} ({CLASS_NAMES[nbr_class]:<25}) α={attn:.4f}  {same}")

> **Key observation:** The GAT systematically assigns higher attention to neighbours that share the **same class** as node 0 — even though class labels are never explicitly provided during aggregation. The model has learned from the training signal that same-class neighbours are more predictive, and routes more information through those edges. This is a non-trivial emergent behaviour: the graph attention mechanism has discovered **homophily** (the tendency for connected nodes to share labels) without being explicitly told about it.

### Exercise 3.3.6 `[Core]` — Compare attention across multiple nodes

Pick 3 nodes from different classes (one from each of: Neural Networks, Genetic Algorithms, Reinforcement Learning). For each node:
1. Draw the attention subgraph.
2. Check whether the highest-attention neighbours share the same class.

Do you observe consistent same-class attention focusing across all three nodes?

In [ ]:
# YOUR CODE HERE
# Hint: Pick 3 nodes from different classes (one from each of: Neural Networks, Genetic Algorithms, Reinforcement Learning). ...
raise NotImplementedError("Complete this exercise")

> **Observation:** Across different nodes and classes, the GAT consistently attends more to same-class neighbours. This is not guaranteed by the architecture — it is a learned behaviour. The attention mechanism has discovered the homophily structure of Cora entirely from the node classification signal.

---
## 4 · `[Extension]` GAT attention vs. Transformer self-attention

In Lab 2.1 you learned that Transformers use **self-attention** to model dependencies between tokens in a sequence:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left( \frac{QK^\top}{\sqrt{d_k}} \right) V$$

In GAT, the attention between nodes $i$ and $j$ is:
$$\alpha_{ji} = \text{softmax}_j\!\left( \text{LeakyReLU}( \mathbf{a}^\top [W h_j \| W h_i] ) \right)$$

Both mechanisms:
1. Compute a **compatibility score** between a query (target) and a key (source).
2. Apply **softmax** to normalise into a valid attention distribution.
3. Compute a **weighted sum** of values (features) weighted by the attention.

The key difference is **connectivity**: Transformer self-attention connects every token to every other token (quadratic in sequence length). GAT attention only connects nodes that are already linked by an edge (sparse — linear in number of edges). This is why GAT scales better to large graphs.

### Exercise 3.3.7 `[Extension]` — Multi-head attention entropy

Different attention heads can learn different patterns. Compute the **attention entropy** for each of the 8 heads:

$$H_h(i) = -\sum_{j \in \mathcal{N}(i)} \alpha_{ji}^{(h)} \log \alpha_{ji}^{(h)}$$

A head with **low entropy** is highly focused (few neighbours receive most attention). A head with **high entropy** distributes attention more uniformly.

1. Compute the mean entropy per head across all nodes.
2. Plot a bar chart of mean entropy per head.
3. Do different heads have different entropy? What might this mean?

In [ ]:
# YOUR CODE HERE
# Hint: Different attention heads can learn different patterns. Compute the attention entropy for each of the 8 heads:
raise NotImplementedError("Complete this exercise")

> **Observation:** The 8 heads exhibit different entropy levels — some are focused (concentrate attention on 1–2 neighbours) while others are more diffuse (spread attention across many neighbours). This is analogous to how different Transformer heads specialise in different syntactic and semantic relations (some track subject-verb agreement, others track coreference). In GAT, different heads may learn to detect different types of topological relationships in the citation network.

---
## 5 · Module III results summary

Let us put all three Module III labs together into a single comparison.

In [ ]:
results = [
    ("MLP (no graph)",   "~0.60", "Node features only; ignores edge structure"),
    ("GCN (2 layers)",   "~0.81", "Symmetric neighbourhood aggregation; fast"),
    ("GAT (8 heads)",    "~0.83", "Learnable per-edge attention; interpretable"),
]

print(f"{'Model':<22} {'Test acc':>10}  Notes")
print("-" * 70)
for model, acc, notes in results:
    print(f"{model:<22} {acc:>10}  {notes}")

---
## Summary

| What we learned | Key takeaway |
|---|---|
| GAT learns per-edge attention $\alpha_{ji}$ | Not all neighbours are equal — the model can focus |
| Multi-head attention finds diverse patterns | Each head specialises in a different neighbourhood relationship |
| Attention is interpretable | Visualising $\alpha_{ji}$ shows *what* the model focused on |
| GAT ≈ GCN with learned weights | Small accuracy gain, large interpretability gain |
| GAT attention mirrors Transformer attention | Same softmax-weighted aggregation — just over a sparse graph |

**Module III complete.** You have:
- Explored the Cora citation network (Lab 3.1).
- Shown that graph structure adds ~20 pp accuracy over feature-only MLP (Lab 3.2).
- Learned that attention makes aggregation selective and interpretable (Lab 3.3).

**Module IV** brings these two worlds together: LLMs and GNNs. We will explore how **graph structure can improve retrieval** (GraphRAG) and how **Transformers can operate directly on graphs** — bridging the two approaches into a unified architecture for reasoning over relational data.